# 26 – Pydantic v2 Advanced Models

## Learning objectives

- Apply field and model validators, computed fields, and Annotated constraints in Pydantic v2.
- Relate framework base models (`AgenticEntity`) to JSON Schema and tool definitions.
- Export OpenAI-style function schemas from Pydantic models for LLM tool calling.

## About this topic

Pydantic v2 separates validation (coercion, constraints) from serialization. Custom validators enforce invariants; computed fields expose derived values in `model_dump` and in generated schemas. Annotated types bundle validators and constraints for reuse. For agents, `model_json_schema()` is the bridge between your Python types and the JSON the model must emit or accept when calling tools.

In [ ]:
from pathlib import Path
import importlib.util
import json

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "pyproject.toml").exists():
        return cwd
    if (cwd.parent / "pyproject.toml").exists():
        return cwd.parent
    raise RuntimeError("Run from repo root or notebooks/")

def load_example(name: str, rel: str):
    path = _repo_root() / rel
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(path)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m

adv = load_example("adv", "examples/library/pydantic_models/advanced_validation.py")
inv = adv.Invoice(
    customer="  demo_user ",
    lines=[adv.InvoiceLine(sku="abc-1", quantity=3, unit_price=adv.Decimal("10.00"))],
)
print("grand_total:", inv.grand_total)
print("dump keys:", sorted(inv.model_dump().keys()))

In [ ]:
from agentic_assistants.core.base_models import AgenticEntity
from pydantic import Field, computed_field

class ToolBackedEntity(AgenticEntity):
    "Domain row that also exposes a derived display name for UIs."
    slug: str = Field(default="", max_length=64)
    title: str = Field(min_length=1)

    @computed_field  # type: ignore[prop-decorator]
    @property
    def display_name(self) -> str:
        base = self.title.strip()
        return f"{base} ({self.slug})" if self.slug else base

row = ToolBackedEntity(slug="rag-01", title="  Notebook RAG  ")
print(row.display_name)
props = row.model_json_schema().get("properties", {})
print("schema sample keys:", sorted(props.keys())[:10])

In [ ]:
import importlib.util
import json
from pathlib import Path

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/pydantic_models/schema_generation.py"
spec = importlib.util.spec_from_file_location("schema_gen", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

tool = mod.openai_function_schema(
    name="get_weather",
    description="Fetch current weather for a city.",
    model=mod.WeatherArgs,
)
print(json.dumps(tool, indent=2)[:1400])

## Exercises and next steps

1. Add a field validator on `ToolBackedEntity.slug` to enforce lowercase kebab-case.
2. Extend `WeatherArgs` in `examples/library/pydantic_models/schema_generation.py` with optional `language: str` and regenerate the tool schema.
3. Sketch how you would map `Invoice` from the example module to a persistence row without leaking `Decimal` to a public JSON API.